<a href="https://colab.research.google.com/github/vinaysampath30-lgtm/gen-ai-skill/blob/main/Gen_AI_Week_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q google-genai

from contextlib import redirect_stdout
import io
import time
from types import SimpleNamespace
from google.colab import userdata
from google import genai
from google.genai import errors, types

SECRET_NAME = "GEMINI_API_KEY"
MODELS = (
    "gemini-3.6-flash",
    "gemini-3.8-flash",
    "gemini-3.5-flash-lite",
    "gemini-flash-latest",
)
TIMEOUT_MS = 20_000
MAX_ROUNDS = 5  # ponytail: fixed 5-round static loop; upgrade to Cloudflare AI Gateway dynamic fallback via HttpOptions(base_url=...) for production load
ROUND_DELAY_SEC = 2.0
CONFIG = types.GenerateContentConfig(
    system_instruction="Answer clearly and concisely in plain language.",
)
PROMPTS = (
    "What is Artificial Intelligence? Explain in simple words.",
    "Explain Generative AI with two real-world examples.",
    "What is a Large Language Model? Explain for a first-year engineering student.",
    "Differentiate Artificial Intelligence, Machine Learning, and Deep Learning.",
)


def _retryable_errors():
    found = [errors.ServerError, TimeoutError]
    for mod_name in ("httpx", "httpx2"):
        try:
            mod = __import__(mod_name)
        except ImportError:
            continue
        for cls_name in ("TimeoutException", "TransportError"):
            cls = getattr(mod, cls_name, None)
            if isinstance(cls, type) and issubclass(cls, BaseException):
                found.append(cls)
    return tuple(dict.fromkeys(found))


def load_api_key(secret_name):
    key = (userdata.get(secret_name) or "").strip().strip("\"'").strip()
    if not key or any(ch.isspace() for ch in key):
        raise ValueError(f"Secret '{secret_name}' is empty or contains whitespace.")
    if key.count("AQ.") > 1 or key.count("AIza") > 1:
        raise ValueError(f"Secret '{secret_name}' appears to contain a duplicated key.")
    return key


def _consume_stream(chunks):
    got_text, usage = False, None
    for chunk in chunks:
        if chunk.text:
            print(chunk.text, end="", flush=True)
            got_text = True
        if chunk.usage_metadata:
            usage = chunk.usage_metadata
    return got_text, usage


def stream_prompt(stream_fn, prompt, models, retryable, max_rounds=MAX_ROUNDS, delay_sec=ROUND_DELAY_SEC):
    active = tuple(models)
    for round_num in range(1, max_rounds + 1):
        for model in active:
            print(f"[Round {round_num} | {model}]", flush=True)
            try:
                got_text, usage = _consume_stream(stream_fn(model, prompt))
                if got_text:
                    reordered = (model,) + tuple(m for m in active if m != model)
                    return model, usage, reordered
                print(" -> Empty response, trying next model...", flush=True)
            except errors.ClientError as e:
                if e.code == 404:
                    print(f" -> Unavailable (404), dropping {model}.", flush=True)
                    active = tuple(m for m in active if m != model)
                elif e.code == 429:
                    print(" -> Rate limited (429), trying next model...", flush=True)
                else:
                    raise
            except retryable as e:
                print(f"\n -> Temporary error ({type(e).__name__}), trying next model...", flush=True)

        if not active:
            raise RuntimeError("No candidate models are available to this API key.")
        if round_num < max_rounds:
            time.sleep(delay_sec)

    raise RuntimeError(f"All models failed after {max_rounds} rounds.")


def _verify_fallback():
    calls = []

    def fake_stream(model, _prompt):
        calls.append(model)
        if model == "m1":
            raise TimeoutError("simulated timeout")
        return (SimpleNamespace(text="ok", usage_metadata="u"),)

    with redirect_stdout(io.StringIO()):
        winner, usage, next_models = stream_prompt(
            fake_stream, "ping", ("m1", "m2"), (TimeoutError,), max_rounds=1, delay_sec=0
        )
    assert (winner, usage, next_models, calls) == ("m2", "u", ("m2", "m1"), ["m1", "m2"])


_verify_fallback()
RETRYABLE = _retryable_errors()
active_models = MODELS

with genai.Client(
    api_key=load_api_key(SECRET_NAME),
    vertexai=False,
    http_options=types.HttpOptions(timeout=TIMEOUT_MS),
) as client:
    call_stream = lambda model, prompt: client.models.generate_content_stream(
        model=model, contents=prompt, config=CONFIG
    )
    for prompt in PROMPTS:
        print(f"Prompt: {prompt}\n", flush=True)
        used_model, usage, active_models = stream_prompt(
            call_stream, prompt, active_models, RETRYABLE
        )
        tokens = (
            f" | Tokens -> input: {usage.prompt_token_count}, output: {usage.candidates_token_count}"
            if usage
            else ""
        )
        print(f"\n\n[Model: {used_model}{tokens}]\n" + "-" * 60, flush=True)

Prompt: What is Artificial Intelligence? Explain in simple words.

[Round 1 | gemini-3.6-flash]
**Artificial Intelligence (AI)** is technology that allows computers and machines to simulate human intelligence. 

In simple terms, it means creating computer systems that can:
* **Learn** from information and past experience.
* **Solve problems** and make decisions.
* **Understand human language** and recognize images or voices.

Unlike regular software, which can only follow strict, step-by-step instructions, AI can adapt, spot patterns, and improve over time—much like a human mind does.

**Everyday examples of AI include:**
* **Voice assistants** like Siri or Alexa.
* **Recommendation systems** on Netflix, YouTube, or Spotify.
* **Navigation apps** like Google Maps that predict traffic.
* **Chatbots** (like ChatGPT) that can answer questions and write text.

[Model: gemini-3.6-flash | Tokens -> input: 20, output: 168]
------------------------------------------------------------
Prompt: E